In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


# 1. Importing necessary libraries

In [ ]:
# 1. Importing necessary libraries
import numpy as np
import pandas as pd

# 2. Loading Datasets

In [ ]:
train_data = pd.read_csv("../input/forest-cover-type-prediction/train.csv")
test_data = pd.read_csv("../input/forest-cover-type-prediction/test.csv")

In [ ]:
train_data.head()

In [ ]:
train_data.info()

In [ ]:
test_data.head()

In [ ]:
test_data.info()

In [ ]:
test_data.isnull().sum()

In [ ]:
test_data.isnull().sum()

In [ ]:
train_data.Cover_Type.value_counts()

In [ ]:
# feature scaling
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [ ]:
scaler.fit(train_data.drop(["Id", "Cover_Type"], axis=1))

In [ ]:
scaled_train_data = scaler.transform(train_data.drop(["Id", "Cover_Type"], axis=1))

In [ ]:
scaled_train_data.shape

In [ ]:
scaled_test_data = scaler.transform(test_data.drop("Id", axis=1))

In [ ]:
scaled_test_data.shape

In [ ]:
X_train = scaled_train_data
y_train = train_data["Cover_Type"].to_numpy()

In [ ]:
X_test = scaled_test_data

# Create neural network

In [ ]:
import tensorflow as tf
from tensorflow import keras


In [ ]:
input_dim = X_train.shape[1] 
hidden_units = [64, 128, 64, 64, 64, 64, 64]  
num_classes = 7  

# Create a Sequential model
model = keras.Sequential()

# Add an input layer
model.add(keras.layers.Input(shape=(input_dim,)))

# Add hidden layers with specified number of neurons and activation functions
for units in hidden_units:
    model.add(keras.layers.Dense(units, activation='relu'))

# Add an output layer without softmax activation
model.add(keras.layers.Dense(num_classes, activation=None))

# Compile the model with an optimizer and custom loss function
model.compile(optimizer='adam', loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=['accuracy'])

# Display the model architecture
model.summary()

In [ ]:
# X_train (training data) and y_train (numeric class labels) ready

history = model.fit(X_train, y_train-1, epochs=100, batch_size=32, validation_split=0.2)


In [ ]:
prediction = model.predict(X_test)


In [ ]:
prediction

In [ ]:
# If you shifted class labels during training, add 1 to the predicted labels
predicted_classes = prediction.argmax(axis=1) + 1

In [ ]:
predicted_classes

In [ ]:
submission = {
    "Id": test_data["Id"],
    "Cover_Type": predicted_classes
}

In [ ]:
submission

In [ ]:
submission = pd.DataFrame(submission)


In [ ]:
submission

In [ ]:
submission["Cover_Type"].value_counts()

In [ ]:
submission.to_csv('submission.csv', index=False)
